# The Pensieve Project — First Investigation

**Objetivo:** conduzir sua primeira investigação reproduzível escolhendo uma trilha: `DFIR`, `OSINT` ou `HYBRID`.

Este notebook usa apenas dados sintéticos pequenos. GLiNER e Qwen ficam desligados por padrão.

Contrato: **evento ≠ interpretação · correlação ≠ causalidade · entidade ≠ identidade · saída de IA ≠ evidência**.


## 0. Instalar o Pensieve

O primeiro caminho instala apenas o core. Modelos opcionais aparecem mais tarde, e só se você ativá-los conscientemente.


In [ ]:
%pip -q install git+https://github.com/Ridd1kulusC0d3r/ThePensieveProject.git

import json, os, subprocess, sys, urllib.request
from pathlib import Path
from IPython.display import HTML, Markdown, display

ROOT = Path('/content/pensieve-first-investigation')
ROOT.mkdir(parents=True, exist_ok=True)
print('Workspace:', ROOT)


## C00 — Escolha a trilha

- `DFIR`: timeline, triage e correlação temporal.
- `OSINT`: entidades, normalização e grafo de coocorrência.
- `HYBRID`: fusão DFIR + OSINT, Evidence Graph e Evidence Packet v2.


In [ ]:
TRACK = 'HYBRID'  # troque para 'DFIR' ou 'OSINT'

TRACK = TRACK.upper().strip()
if TRACK not in {'DFIR', 'OSINT', 'HYBRID'}:
    raise ValueError('TRACK deve ser DFIR, OSINT ou HYBRID')

DATASETS = {
    'DFIR': 'dfir.csv',
    'OSINT': 'osint.csv',
    'HYBRID': 'hybrid.csv',
}
BASE = 'https://raw.githubusercontent.com/Ridd1kulusC0d3r/ThePensieveProject/main/examples/first-investigation'
dataset_name = DATASETS[TRACK]
evidence = ROOT / 'evidence.csv'
urllib.request.urlretrieve(f'{BASE}/{dataset_name}', evidence)

progress = {
    'schema_version': '1.0',
    'track': TRACK,
    'dataset': dataset_name,
    'evidence_sha256': '',
    'missions': ['C00'],
    'artifacts': ['evidence.csv'],
    'optional_models': {'gliner': False, 'qwen': False},
    'notes': {},
}
print('Trilha:', TRACK)
print('Dataset:', evidence)


## C01 — Preservar antes de interpretar

Execute `doctor` e calcule o SHA-256 antes da análise. O hash ajuda a demonstrar que você está trabalhando sobre os mesmos bytes; ele **não** prova autoria ou cadeia de custódia completa.


In [ ]:
subprocess.run(['pensieve-timeline', 'doctor'], check=True)
hash_output = subprocess.check_output(
    ['pensieve-timeline', 'hash', str(evidence)], text=True
)
hash_info = json.loads(hash_output)
progress['evidence_sha256'] = hash_info['digest']
progress['missions'].append('C01')
print('SHA-256:', progress['evidence_sha256'])


### Checkpoint C01

Antes de seguir, responda mentalmente ou em suas notas:

1. O que o SHA-256 prova neste laboratório?
2. O que ele não prova?
3. Qual arquivo está sendo tratado como evidência de entrada?


## C02 — Criar a primeira timeline

A ingestão converte registros heterogêneos em `ForensicEvent`, mantendo proveniência e o registro original em camada separada.


In [ ]:
timeline = ROOT / 'timeline.jsonl'
case_db = ROOT / 'case.db'
subprocess.run([
    'pensieve-timeline', 'ingest', str(evidence),
    '-o', str(timeline), '--sqlite', str(case_db)
], check=True)

events = [json.loads(line) for line in timeline.read_text(encoding='utf-8').splitlines() if line.strip()]
progress['missions'].append('C02')
progress['artifacts'] += ['timeline.jsonl', 'case.db']

print('Eventos:', len(events))
for row in events[:3]:
    print({
        'event_id': row['event_id'],
        'timestamp': row['timestamp'],
        'source': row['source'],
        'event_code': row.get('event_code'),
        'risk_score': row['risk_score'],
        'record_locator': row['record_locator'],
    })


### Checkpoint C02

Escolha um `event_id` exibido acima e localize o timestamp, a fonte e o `record_locator`. Você só avança quando consegue explicar como esse evento chegou à timeline.


## Executar a trilha escolhida

A célula abaixo muda o pipeline conforme a trilha. Ela produz artefatos diferentes, mas mantém o mesmo princípio: **resultado derivado nunca substitui evidência**.


In [ ]:
def run(*args):
    cmd = ['pensieve-timeline', *map(str, args)]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

report = ROOT / 'report.html'

if TRACK == 'DFIR':
    correlations = ROOT / 'correlations.jsonl'
    run('correlate', timeline, '-o', correlations, '--window', '120')
    run('report', timeline, '-o', report)
    progress['missions'] += ['D01', 'D02', 'D03']
    progress['artifacts'] += ['correlations.jsonl', 'report.html']

elif TRACK == 'OSINT':
    entities = ROOT / 'entities.jsonl'
    graph = ROOT / 'entity-graph.json'
    run('entities', timeline, '-o', entities)
    run('graph', entities, '-o', graph)
    run('report', timeline, '--entities', entities, '-o', report)
    progress['missions'] += ['O01', 'O02']
    progress['artifacts'] += ['entities.jsonl', 'entity-graph.json', 'report.html']

else:
    entities = ROOT / 'entities.jsonl'
    graph = ROOT / 'evidence-graph.json'
    packet = ROOT / 'evidence-packet.json'
    run('entities', timeline, '-o', entities)
    run('evidence-graph', timeline, entities, '-o', graph)
    run('evidence-packet', timeline, '--entities', entities, '-o', packet)
    run('report', timeline, '--entities', entities, '-o', report)
    progress['missions'] += ['H01', 'H02']
    progress['artifacts'] += ['entities.jsonl', 'evidence-graph.json', 'evidence-packet.json', 'report.html']

print('\nArtefatos produzidos:')
for name in progress['artifacts']:
    print(' -', name)


In [ ]:
if TRACK == 'DFIR':
    prompt = '''### Checkpoint DFIR\n1. Qual evento você priorizaria e por quê?\n2. Qual alternativa benigna continua plausível?\n3. Escolha uma correlação e liste apenas as regras que a criaram.\n4. Escreva uma hipótese com event_ids e uma alternativa.'''
elif TRACK == 'OSINT':
    prompt = '''### Checkpoint OSINT\n1. Escolha uma entidade e compare text vs normalized_value.\n2. O que evidence_hash ajuda a auditar?\n3. Escolha uma aresta de coocorrência. O que ela prova?\n4. O que seria necessário para afirmar identidade ou controle?'''
else:
    prompt = '''### Checkpoint DFIR + OSINT\n1. Encontre um event node e um entity node no Evidence Graph.\n2. Diferencie mentioned_in de correlated_with.\n3. Abra o Evidence Packet e confira packet_sha256.\n4. Liste o que foi deliberadamente excluído do boundary de IA.'''
display(Markdown(prompt))


## Inspeção rápida dos resultados

A célula não decide o caso por você. Ela só resume contagens para orientar onde olhar.


In [ ]:
summary = {'events': len(events)}
if (ROOT / 'correlations.jsonl').exists():
    summary['correlations'] = len([x for x in (ROOT/'correlations.jsonl').read_text().splitlines() if x.strip()])
if (ROOT / 'entities.jsonl').exists():
    entity_rows = [json.loads(x) for x in (ROOT/'entities.jsonl').read_text().splitlines() if x.strip()]
    summary['entity_mentions'] = len(entity_rows)
    summary['entity_labels'] = sorted({x['label'] for x in entity_rows})
if (ROOT / 'evidence-packet.json').exists():
    packet_data = json.loads((ROOT/'evidence-packet.json').read_text())
    summary['packet_id'] = packet_data['packet_id']
    summary['packet_sha256'] = packet_data['packet_sha256']
print(json.dumps(summary, indent=2, ensure_ascii=False))


## GLiNER opcional — somente OSINT/HYBRID

Ative apenas depois de entender a extração determinística. O modelo adiciona capacidade de NER; não adiciona verdade automática.


In [ ]:
USE_GLINER = False

if USE_GLINER and TRACK in {'OSINT', 'HYBRID'}:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gliner>=0.2'], check=True)
    gliner_output = ROOT / 'entities-gliner.jsonl'
    run('entities', timeline, '--gliner',
        '--labels', 'person', 'organization', 'email', 'domain', 'social media handle',
        '-o', gliner_output)
    progress['optional_models']['gliner'] = True
    progress['artifacts'].append('entities-gliner.jsonl')
    if 'O03' not in progress['missions']:
        progress['missions'].append('O03' if TRACK == 'OSINT' else 'H03-GLINER')
else:
    print('GLiNER não executado. Isso é o padrão recomendado para a primeira passagem.')


## Qwen opcional — somente HYBRID

O Pensieve gera o Evidence Packet antes do modelo, exige citações por `event_id` e rejeita campos que tentem reescrever a evidência.


In [ ]:
USE_QWEN = False

if USE_QWEN and TRACK == 'HYBRID':
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers>=4.51,<5', 'accelerate>=1,<2', 'torch>=2.6'
    ], check=True)
    intelligence = ROOT / 'intelligence.json'
    run('reason', timeline, ROOT/'entities.jsonl',
        '--packet-output', ROOT/'evidence-packet.json',
        '--model', 'Qwen/Qwen3-0.6B', '-o', intelligence)
    progress['optional_models']['qwen'] = True
    progress['missions'].append('H03')
    progress['artifacts'].append('intelligence.json')
    print(intelligence.read_text(encoding='utf-8')[:5000])
else:
    print('Qwen não executado. Evidence Packet v2 continua disponível para auditoria.')


## Dashboard local

Use o relatório para explorar a timeline. O dashboard é uma interface de leitura; ele não substitui suas notas investigativas.


In [ ]:
display(HTML(report.read_text(encoding='utf-8')))


## Capstone — escreva antes de pedir ajuda à IA

Registre:

1. **3 observações** literais com `event_id`.
2. **1 inferência** e o raciocínio que a sustenta.
3. **1 hipótese** com pelo menos uma alternativa plausível.
4. **2 incertezas** que continuam abertas.
5. **1 próxima coleta** que reduziria a maior incerteza.

A investigação melhora quando você consegue mostrar onde pode estar errado.


In [ ]:
progress_path = ROOT / 'academy-progress.json'
progress['missions'] = list(dict.fromkeys(progress['missions']))
progress['artifacts'] = list(dict.fromkeys(progress['artifacts']))
progress_path.write_text(json.dumps(progress, indent=2, ensure_ascii=False), encoding='utf-8')

import shutil
archive = shutil.make_archive('/content/pensieve-first-investigation', 'zip', ROOT)
print('Progress:', progress_path)
print('Archive:', archive)
print(json.dumps(progress, indent=2, ensure_ascii=False))


## Próximo passo

Depois de concluir uma trilha, execute outra usando o mesmo método. Só então avance para os LABs pesados de EVTX, Registry, MFT/USN e imagens forenses. Formato binário novo não conserta raciocínio ruim; apenas o torna mais caro.
